VirulentPred 2.0 Database Update Pipeline - File-by-File Processing
=====================================================================
This notebook processes hypothetical protein sequences ONE FILE AT A TIME,
filters duplicates against existing databases, predicts virulence using
VirulentPred 2.0 standalone, and updates databases AFTER EACH FILE.

Workflow (for EACH Excel file):
1. Extract sequences from Excel file (column O: aa_sequence)
2. Align against current DB.fasta + Non-DB.fasta using DIAMOND
3. Discard 100% identical sequences (identity=100, gaps=0, query_coverage=100%)
4. Run VirulentPred 2.0 standalone on remaining sequences
5. Update DB.fasta and Non-DB.fasta immediately
6. Move to next Excel file (databases are now updated)

In [ ]:
# ============================================================================
# CELL 1: Setup - Install DIAMOND
# ============================================================================
# @title Cell 1: Install DIAMOND

import os
import sys

print("="*80)
print("SETUP: INSTALLING DIAMOND")
print("="*80)

if not os.path.exists('/opt/conda/bin/mamba'):
    print('\n📥 Installing mambaforge...')
    !wget -q https://github.com/conda-forge/miniforge/releases/download/24.3.0-0/Mambaforge-24.3.0-0-Linux-x86_64.sh -O mamba.sh
    !bash mamba.sh -u -b -p /opt/conda
    !rm mamba.sh
    os.environ["PATH"] = "/opt/conda/bin:" + os.environ["PATH"]

os.environ["PATH"] = "/opt/conda/bin:" + os.environ["PATH"]

print("\n📥 Installing DIAMOND...")
!mamba install -c bioconda diamond -y -q
print("✅ DIAMOND installed")
!diamond --version

print("\n✅ Setup complete!")
print("="*80)

In [ ]:
# ============================================================================
# CELL 2: Mount Google Drive and Setup Paths
# ============================================================================
# @title Cell 2: Mount Drive and Configure Paths

from google.colab import drive
import os

print("="*80)
print("GOOGLE DRIVE SETUP")
print("="*80)

drive.mount('/content/drive')
print("✅ Google Drive mounted")

# ===== CONFIGURATION - UPDATE THESE PATHS =====
DB_BASE_PATH = "/content/drive/MyDrive/VirulentPred_Databases"
INPUT_PATH = "/content/drive/MyDrive/VirulentPred_Input"

os.makedirs(DB_BASE_PATH, exist_ok=True)
os.makedirs(INPUT_PATH, exist_ok=True)

# Database paths
DB_VIRULENT = os.path.join(DB_BASE_PATH, "DB.fasta")
DB_NON_VIRULENT = os.path.join(DB_BASE_PATH, "Non-DB.fasta")

print(f"\n📁 Database path: {DB_BASE_PATH}")
print(f"📁 Input path: {INPUT_PATH}")

# Initialize databases if needed
for db in [DB_VIRULENT, DB_NON_VIRULENT]:
    if not os.path.exists(db):
        open(db, 'w').close()

print("\n✅ Paths configured!")
print("="*80)

In [ ]:
# ============================================================================
# CELL 3: MAIN PIPELINE - Collect All Novel Sequences
# ============================================================================
# @title Cell 3: Collect Novel Sequences from All Excel Files

import pandas as pd
import glob
import re
from datetime import datetime
from collections import OrderedDict

print("="*80)
print("NOVEL SEQUENCE COLLECTION PIPELINE")
print("="*80)
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ===== CONFIGURATION =====
FILE_PATTERN = "Hypothetical protein-Filtered-Erwinia amylovora ID*.xlsx"
SHEET_NAME = "Hypothetical_Proteins"  # Update based on your actual sheet name
SEQUENCE_COLUMN = "aa_sequence"

print(f"\n⚙️ Configuration:")
print(f"   • Input: {INPUT_PATH}")
print(f"   • Database: {DB_BASE_PATH}")
print(f"   • Sheet: '{SHEET_NAME}'")
print(f"   • Column: '{SEQUENCE_COLUMN}'")
print("="*80)

# ===== HELPER FUNCTIONS =====

def count_seqs(fasta_file):
    """Count sequences in FASTA"""
    if not os.path.exists(fasta_file) or os.path.getsize(fasta_file) == 0:
        return 0
    with open(fasta_file) as f:
        return sum(1 for line in f if line.startswith('>'))

def read_fasta(fasta_file):
    """Read FASTA into dictionary"""
    seqs = {}
    if not os.path.exists(fasta_file) or os.path.getsize(fasta_file) == 0:
        return seqs

    with open(fasta_file) as f:
        header = None
        seq = []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header:
                    seqs[header] = ''.join(seq)
                header = line[1:]
                seq = []
            else:
                seq.append(line)
        if header:
            seqs[header] = ''.join(seq)
    return seqs

def write_fasta(sequences_dict, output_file):
    """Write dictionary to FASTA"""
    with open(output_file, 'w') as f:
        for header, seq in sequences_dict.items():
            f.write(f">{header}\n{seq}\n")

def create_combined_db():
    """Combine DB.fasta + Non-DB.fasta for DIAMOND"""
    combined = os.path.join(DB_BASE_PATH, "Combined-DB.fasta")

    seqs_vir = read_fasta(DB_VIRULENT)
    seqs_nonvir = read_fasta(DB_NON_VIRULENT)
    all_seqs = {**seqs_vir, **seqs_nonvir}

    if all_seqs:
        write_fasta(all_seqs, combined)
        cmd = f"diamond makedb --in '{combined}' -d '{DB_BASE_PATH}/Combined-DB' --quiet"
        os.system(cmd)
        return True
    return False

def run_diamond_filter(query_fasta):
    """Run DIAMOND and return duplicate IDs"""
    output = query_fasta.replace('.fasta', '_diamond.tsv')

    if not os.path.exists(f"{DB_BASE_PATH}/Combined-DB.dmnd"):
        return set()

    cmd = f"diamond blastp -d '{DB_BASE_PATH}/Combined-DB' -q '{query_fasta}' -o '{output}' "
    cmd += "--outfmt 6 qseqid sseqid pident gaps qlen length --quiet --max-target-seqs 1"
    os.system(cmd)

    duplicates = set()
    if os.path.exists(output) and os.path.getsize(output) > 0:
        with open(output) as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 6:
                    qid = parts[0]
                    identity = float(parts[2])
                    gaps = int(parts[3])
                    qlen = int(parts[4])
                    length = int(parts[5])
                    coverage = (length / qlen) * 100

                    # 100% match criteria
                    if identity == 100.0 and gaps == 0 and coverage >= 100.0:
                        duplicates.add(qid)

    return duplicates

def extract_id(filename):
    """Extract ID from filename"""
    match = re.search(r'ID(\d+)', filename)
    return match.group(1) if match else None

# ===== STEP 1: BUILD COMBINED DATABASE =====
print("\n" + "="*80)
print("STEP 1: Building Combined Database")
print("="*80)

db_exists = create_combined_db()

vir_count = count_seqs(DB_VIRULENT)
nonvir_count = count_seqs(DB_NON_VIRULENT)

print(f"\n📊 Current databases:")
print(f"   • DB.fasta: {vir_count} sequences")
print(f"   • Non-DB.fasta: {nonvir_count} sequences")
print(f"   • Total: {vir_count + nonvir_count} sequences")

if db_exists:
    print(f"✅ Combined DIAMOND database built")
else:
    print(f"⚠️  Databases are empty (all sequences will be novel)")

# ===== STEP 2: COLLECT NOVEL SEQUENCES FROM ALL FILES =====
print("\n" + "="*80)
print("STEP 2: Collecting Novel Sequences from All Excel Files")
print("="*80)

# Find all Excel files
excel_files = sorted(
    glob.glob(os.path.join(INPUT_PATH, FILE_PATTERN)),
    key=lambda x: int(extract_id(x) or 0)
)

if not excel_files:
    print(f"\n❌ No files found matching: {FILE_PATTERN}")
    print(f"   in directory: {INPUT_PATH}")
else:
    print(f"\n✅ Found {len(excel_files)} Excel files\n")

    # Collection dictionary: sequence -> first occurrence info
    all_novel_sequences = OrderedDict()

    # Statistics
    stats = {
        'total_seqs': 0,
        'duplicates_vs_db': 0,
        'novel': 0,
        'failed': 0
    }

    # Process each file
    for idx, excel_file in enumerate(excel_files, 1):
        try:
            file_id = extract_id(os.path.basename(excel_file))
            if not file_id:
                continue

            print(f"[{idx}/{len(excel_files)}] ID{file_id}...", end=' ')

            # Read Excel
            try:
                df = pd.read_excel(excel_file, sheet_name=SHEET_NAME)
            except ValueError:
                xl = pd.ExcelFile(excel_file)
                df = pd.read_excel(excel_file, sheet_name=xl.sheet_names[0])

            # Check column exists
            if SEQUENCE_COLUMN not in df.columns:
                print(f"❌ Column '{SEQUENCE_COLUMN}' not found")
                stats['failed'] += 1
                continue

            # Extract sequences
            sequences = {}
            for i, row in df.iterrows():
                seq = str(row[SEQUENCE_COLUMN]).strip()
                if seq and seq != 'nan' and len(seq) > 10:
                    seq_id = f"ID{file_id}_row{i+1}"
                    sequences[seq_id] = seq

            stats['total_seqs'] += len(sequences)

            if not sequences:
                print(f"⚠️ No sequences")
                continue

            # Write query FASTA
            query_fasta = f"/content/temp_query_ID{file_id}.fasta"
            write_fasta(sequences, query_fasta)

            # Filter against database
            if db_exists:
                duplicates = run_diamond_filter(query_fasta)
                novel = {k: v for k, v in sequences.items() if k not in duplicates}
                stats['duplicates_vs_db'] += len(duplicates)
            else:
                novel = sequences

            stats['novel'] += len(novel)

            # Add to collection
            for seq_id, seq in novel.items():
                all_novel_sequences[seq_id] = seq

            print(f"✅ {len(sequences)} seqs, {len(novel)} novel")

            # Cleanup
            if os.path.exists(query_fasta):
                os.remove(query_fasta)
            diamond_out = query_fasta.replace('.fasta', '_diamond.tsv')
            if os.path.exists(diamond_out):
                os.remove(diamond_out)

        except Exception as e:
            print(f"❌ Error: {e}")
            stats['failed'] += 1

    # ===== STEP 3: REMOVE INTERNAL DUPLICATES =====
    print("\n" + "="*80)
    print("STEP 3: Removing Internal Duplicates")
    print("="*80)

    print(f"\n📊 Before deduplication: {len(all_novel_sequences)} sequences")

    # Use sequence as key to find duplicates
    seq_to_first_id = {}
    unique_sequences = OrderedDict()
    internal_duplicates = 0

    for seq_id, seq in all_novel_sequences.items():
        if seq not in seq_to_first_id:
            # First occurrence of this sequence
            seq_to_first_id[seq] = seq_id
            unique_sequences[seq_id] = seq
        else:
            # Duplicate sequence - keep the first one
            internal_duplicates += 1

    print(f"📊 After deduplication: {len(unique_sequences)} unique sequences")
    print(f"🔄 Removed {internal_duplicates} internal duplicates")

    # ===== STEP 4: SAVE FINAL FASTA =====
    print("\n" + "="*80)
    print("STEP 4: Saving Final FASTA File")
    print("="*80)

    output_fasta = os.path.join(DB_BASE_PATH, "Novel_Sequences_For_VirulentPred.fasta")
    write_fasta(unique_sequences, output_fasta)

    print(f"\n✅ Saved: {output_fasta}")
    print(f"📊 Contains: {len(unique_sequences)} unique sequences")

    # ===== FINAL SUMMARY =====
    print("\n" + "="*80)
    print("PIPELINE COMPLETE - SUMMARY")
    print("="*80)
    print(f"\nFinished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    print(f"\n📊 Statistics:")
    print(f"   • Excel files processed: {len(excel_files) - stats['failed']}/{len(excel_files)}")
    print(f"   • Total sequences extracted: {stats['total_seqs']:,}")
    print(f"   • Duplicates vs DB.fasta/Non-DB.fasta: {stats['duplicates_vs_db']:,}")
    print(f"   • Novel sequences (before internal dedup): {stats['novel']:,}")
    print(f"   • Internal duplicates removed: {internal_duplicates:,}")
    print(f"   • Final unique novel sequences: {len(unique_sequences):,}")

    print(f"\n📁 Output file: {output_fasta}")
    print(f"📏 File size: {os.path.getsize(output_fasta) / (1024*1024):.2f} MB")

    print("\n" + "="*80)
    print("NEXT STEPS:")
    print("="*80)
    print(f"\n1️⃣ Download the FASTA file:")
    print(f"   {output_fasta}")

    print(f"\n2️⃣ Go to VirulentPred 2.0 web interface:")
    print(f"   https://bioinfo.icgeb.res.in/virulent2/predict.html")

    print(f"\n3️⃣ Submit the FASTA file for prediction")

    print(f"\n4️⃣ Download the results")

    print(f"\n5️⃣ Run Cell 4 (below) to update databases with predictions")

    print("\n" + "="*80)

In [ ]:
# ============================================================================
# CELL 4: Process VirulentPred Results and Update Databases
# ============================================================================
# @title Cell 4: Process VirulentPred Results & Update Databases

import pandas as pd
import os
import glob
from datetime import datetime
from collections import defaultdict

print("="*80)
print("VIRULENTPRED RESULTS PROCESSING & DATABASE UPDATE")
print("="*80)
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ===== CONFIGURATION =====
RESULTS_PATTERN = "VP_2.0_pred_res_*.xls*"  # Matches .xls and .xlsx
RESULTS_DIR = "/content/drive/MyDrive/VirulentPred_Results"  # Update this path!
# Note: DB_BASE_PATH must be defined in previous cells.
# If running standalone, uncomment and set the line below:
DB_BASE_PATH = "/content/drive/MyDrive/VirulentPred_Databases"
NOVEL_FASTA = os.path.join(DB_BASE_PATH, "Novel_Sequences_For_VirulentPred.fasta")

# These should have been defined in previous cells, but defining here for safety if missing
if 'DB_VIRULENT' not in locals():
    DB_VIRULENT = os.path.join(DB_BASE_PATH, "DB.fasta")
if 'DB_NON_VIRULENT' not in locals():
    DB_NON_VIRULENT = os.path.join(DB_BASE_PATH, "Non-DB.fasta")

print(f"\n⚙️ Configuration:")
print(f"    • Results directory: {RESULTS_DIR}")
print(f"    • Results pattern: {RESULTS_PATTERN}")
print(f"    • Novel sequences: {NOVEL_FASTA}")
print(f"    • Database directory: {DB_BASE_PATH}")
print("="*80)

# ===== HELPER FUNCTIONS =====

def read_fasta(fasta_file):
    """Read FASTA file into dictionary {header: sequence}"""
    seqs = {}
    if not os.path.exists(fasta_file) or os.path.getsize(fasta_file) == 0:
        return seqs

    with open(fasta_file) as f:
        header = None
        seq = []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header:
                    seqs[header] = ''.join(seq)
                header = line[1:]  # Remove '>'
                seq = []
            else:
                seq.append(line)
        if header:
            seqs[header] = ''.join(seq)
    return seqs

def count_seqs(fasta_file):
    """Count sequences in FASTA file"""
    if not os.path.exists(fasta_file) or os.path.getsize(fasta_file) == 0:
        return 0
    with open(fasta_file) as f:
        return sum(1 for line in f if line.startswith('>'))

def clean_seq_id(seq_id):
    """Clean sequence ID - remove '>' if present"""
    seq_id = str(seq_id).strip()
    if seq_id.startswith('>'):
        return seq_id[1:]
    return seq_id

# ===== STEP 1: LOAD NOVEL SEQUENCES =====
print("\n" + "="*80)
print("STEP 1: Loading Novel Sequences")
print("="*80)

if not os.path.exists(NOVEL_FASTA):
    print(f"\n❌ Novel sequences file not found: {NOVEL_FASTA}")
    print("\n💡 Please ensure Cell 3 was run successfully")
    import sys
    sys.exit(1)

print(f"\n📂 Reading: {os.path.basename(NOVEL_FASTA)}")
novel_sequences = read_fasta(NOVEL_FASTA)

print(f"✅ Loaded {len(novel_sequences):,} sequences")
print(f"📏 Total size: {sum(len(seq) for seq in novel_sequences.values()):,} amino acids")

# Show sample
if novel_sequences:
    sample_id = list(novel_sequences.keys())[0]
    sample_seq = novel_sequences[sample_id]
    print(f"\n📋 Sample:")
    print(f"    ID: {sample_id}")
    print(f"    Sequence: {sample_seq[:60]}{'...' if len(sample_seq) > 60 else ''}")
    print(f"    Length: {len(sample_seq)} aa")

# ===== STEP 2: FIND RESULT FILES =====
print("\n" + "="*80)
print("STEP 2: Finding VirulentPred Result Files")
print("="*80)

# Check if results directory exists
if not os.path.exists(RESULTS_DIR):
    print(f"\n⚠️ Results directory not found: {RESULTS_DIR}")
    print("\n💡 Creating directory...")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print("✅ Directory created - please upload your 14 result files here")
    import sys
    sys.exit(1)

# Find all result files
result_files = sorted(glob.glob(os.path.join(RESULTS_DIR, RESULTS_PATTERN)))

if not result_files:
    print(f"\n❌ No result files found matching: {RESULTS_PATTERN}")
    print(f"    in directory: {RESULTS_DIR}")
    print(f"\n📁 Files in directory:")
    all_files = os.listdir(RESULTS_DIR)
    if all_files:
        for f in all_files[:20]:
            print(f"    • {f}")
    else:
        print("    (directory is empty)")
    print("\n💡 Please upload your 14 VirulentPred result files to this directory")
    import sys
    sys.exit(1)

print(f"\n✅ Found {len(result_files)} result files:")
for i, rf in enumerate(result_files, 1):
    size = os.path.getsize(rf)
    print(f"    {i:2d}. {os.path.basename(rf):50s} ({size:,} bytes)")

# ===== STEP 3: PARSE ALL RESULT FILES =====
print("\n" + "="*80)
print("STEP 3: Parsing VirulentPred Results")
print("="*80)

virulent_predictions = {}      # {seq_id: True}
non_virulent_predictions = {}  # {seq_id: True}
parsing_errors = []
total_predictions = 0

for i, result_file in enumerate(result_files, 1):
    print(f"\n📄 [{i}/{len(result_files)}] Processing: {os.path.basename(result_file)}")

    try:
        # Read Excel file
        df = pd.read_excel(result_file)

        print(f"    📊 Shape: {df.shape}")
        print(f"    📊 Columns: {list(df.columns)}")

        # Identify columns
        if len(df.columns) < 2:
            print(f"    ⚠️ Expected 2 columns, found {len(df.columns)}")
            parsing_errors.append(f"{os.path.basename(result_file)}: Not enough columns")
            continue

        # Use first two columns (regardless of names)
        id_col = df.columns[0]
        pred_col = df.columns[1]

        print(f"    🔍 Using columns: '{id_col}' and '{pred_col}'")

        # Parse predictions
        chunk_virulent = 0
        chunk_non_virulent = 0
        chunk_unknown = 0

        for idx, row in df.iterrows():
            seq_id = clean_seq_id(row[id_col])
            prediction = str(row[pred_col]).strip()

            # Skip empty rows
            if not seq_id or seq_id == 'nan':
                continue

            total_predictions += 1

            # Classify prediction
            pred_lower = prediction.lower()
            if 'virulent' in pred_lower and 'non' not in pred_lower:
                virulent_predictions[seq_id] = True
                chunk_virulent += 1
            elif 'non-virulent' in pred_lower or 'nonvirulent' in pred_lower:
                non_virulent_predictions[seq_id] = True
                chunk_non_virulent += 1
            else:
                chunk_unknown += 1
                print(f"      ⚠️ Unknown prediction for {seq_id}: '{prediction}'")

        print(f"    ✅ Parsed: {chunk_virulent} virulent, {chunk_non_virulent} non-virulent", end='')
        if chunk_unknown > 0:
            print(f", {chunk_unknown} unknown")
        else:
            print()

    except Exception as e:
        print(f"    ❌ Error: {e}")
        parsing_errors.append(f"{os.path.basename(result_file)}: {str(e)}")

# Summary
print(f"\n" + "="*80)
print("PARSING SUMMARY")
print("="*80)
print(f"\n📊 Total predictions parsed: {total_predictions:,}")
print(f"    • Virulent: {len(virulent_predictions):,} ({len(virulent_predictions)/total_predictions*100:.1f}%)")
print(f"    • Non-virulent: {len(non_virulent_predictions):,} ({len(non_virulent_predictions)/total_predictions*100:.1f}%)")

if parsing_errors:
    print(f"\n⚠️ Parsing errors ({len(parsing_errors)}):")
    for error in parsing_errors:
        print(f"    • {error}")

# Validation check
if total_predictions != len(novel_sequences):
    print(f"\n⚠️ WARNING: Count mismatch!")
    print(f"    Expected: {len(novel_sequences):,} sequences")
    print(f"    Got: {total_predictions:,} predictions")
    diff = abs(len(novel_sequences) - total_predictions)
    print(f"    Difference: {diff:,} sequences")

# ===== STEP 4: MATCH SEQUENCES =====
print("\n" + "="*80)
print("STEP 4: Matching Predictions with Sequences")
print("="*80)

virulent_with_seqs = {}      # {seq_id: sequence}
non_virulent_with_seqs = {}  # {seq_id: sequence}
missing_sequences = []

print(f"\n🔍 Matching virulent predictions...")
for seq_id in virulent_predictions:
    if seq_id in novel_sequences:
        virulent_with_seqs[seq_id] = novel_sequences[seq_id]
    else:
        missing_sequences.append((seq_id, 'virulent'))

print(f"    ✅ Matched: {len(virulent_with_seqs):,}/{len(virulent_predictions):,}")

print(f"\n🔍 Matching non-virulent predictions...")
for seq_id in non_virulent_predictions:
    if seq_id in novel_sequences:
        non_virulent_with_seqs[seq_id] = novel_sequences[seq_id]
    else:
        missing_sequences.append((seq_id, 'non-virulent'))

print(f"    ✅ Matched: {len(non_virulent_with_seqs):,}/{len(non_virulent_predictions):,}")

if missing_sequences:
    print(f"\n⚠️ Missing sequences: {len(missing_sequences)}")
    print(f"    (These IDs were in results but not in FASTA file)")
    if len(missing_sequences) <= 10:
        for seq_id, pred_type in missing_sequences:
            print(f"      • {seq_id} ({pred_type})")
    else:
        print(f"    Showing first 10:")
        for seq_id, pred_type in missing_sequences[:10]:
            print(f"      • {seq_id} ({pred_type})")

# ===== STEP 5: UPDATE DATABASES =====
print("\n" + "="*80)
print("STEP 5: Updating Databases")
print("="*80)

# Count before update
vir_before = count_seqs(DB_VIRULENT)
nonvir_before = count_seqs(DB_NON_VIRULENT)

print(f"\n📊 Database status BEFORE update:")
print(f"    • DB.fasta: {vir_before:,} sequences")
print(f"    • Non-DB.fasta: {nonvir_before:,} sequences")
print(f"    • Total: {vir_before + nonvir_before:,} sequences")

# Append virulent sequences
print(f"\n💾 Updating DB.fasta...")
with open(DB_VIRULENT, 'a') as f:
    for seq_id, sequence in virulent_with_seqs.items():
        f.write(f">{seq_id}\n{sequence}\n")

print(f"    ✅ Added {len(virulent_with_seqs):,} virulent sequences")

# Append non-virulent sequences
print(f"\n💾 Updating Non-DB.fasta...")
with open(DB_NON_VIRULENT, 'a') as f:
    for seq_id, sequence in non_virulent_with_seqs.items():
        f.write(f">{seq_id}\n{sequence}\n")

print(f"    ✅ Added {len(non_virulent_with_seqs):,} non-virulent sequences")

# Count after update
vir_after = count_seqs(DB_VIRULENT)
nonvir_after = count_seqs(DB_NON_VIRULENT)

print(f"\n📊 Database status AFTER update:")
print(f"    • DB.fasta: {vir_after:,} sequences (+{vir_after - vir_before:,})")
print(f"    • Non-DB.fasta: {nonvir_after:,} sequences (+{nonvir_after - nonvir_before:,})")
print(f"    • Total: {vir_after + nonvir_after:,} sequences")

# ===== STEP 6: CREATE DETAILED REPORT =====
print("\n" + "="*80)
print("STEP 6: Creating Detailed Report")
print("="*80)

# Create comprehensive report DataFrame
report_data = []

for seq_id in novel_sequences:
    prediction = None
    if seq_id in virulent_with_seqs:
        prediction = "Virulent"
    elif seq_id in non_virulent_with_seqs:
        prediction = "Non-virulent"
    else:
        prediction = "Missing"

    sequence = novel_sequences[seq_id]

    report_data.append({
        'Sequence_ID': seq_id,
        'Prediction': prediction,
        'Sequence_Length': len(sequence),
        'Sequence': sequence
    })

report_df = pd.DataFrame(report_data)

# Save report
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
report_file = os.path.join(DB_BASE_PATH, f"VirulentPred_Complete_Report_{timestamp}.xlsx")

print(f"\n💾 Saving comprehensive report...")
report_df.to_excel(report_file, index=False)
print(f"    ✅ Saved: {os.path.basename(report_file)}")

# Create statistics summary
stats_data = {
    'Metric': [
        'Total Novel Sequences',
        'Virulent Predictions',
        'Non-virulent Predictions',
        'Missing Predictions',
        'Virulent Percentage',
        'DB.fasta (Before)',
        'DB.fasta (After)',
        'Non-DB.fasta (Before)',
        'Non-DB.fasta (After)',
        'Total Database Size'
    ],
    'Value': [
        len(novel_sequences),
        len(virulent_with_seqs),
        len(non_virulent_with_seqs),
        len(missing_sequences),
        f"{len(virulent_with_seqs)/len(novel_sequences)*100:.2f}%",
        vir_before,
        vir_after,
        nonvir_before,
        nonvir_after,
        vir_after + nonvir_after
    ]
}

stats_df = pd.DataFrame(stats_data)
stats_file = os.path.join(DB_BASE_PATH, f"VirulentPred_Statistics_{timestamp}.xlsx")

print(f"\n💾 Saving statistics summary...")
with pd.ExcelWriter(stats_file) as writer:
    stats_df.to_excel(writer, sheet_name='Statistics', index=False)
    report_df.to_excel(writer, sheet_name='All_Results', index=False)

print(f"    ✅ Saved: {os.path.basename(stats_file)}")

# ===== FINAL SUMMARY =====
print("\n" + "="*80)
print("PIPELINE COMPLETE - FINAL SUMMARY")
print("="*80)
print(f"\nFinished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n🎯 Processing Summary:")
print(f"    • Result files processed: {len(result_files)}")
print(f"    • Novel sequences: {len(novel_sequences):,}")
print(f"    • Total predictions: {total_predictions:,}")
print(f"    • Virulent: {len(virulent_with_seqs):,} ({len(virulent_with_seqs)/len(novel_sequences)*100:.1f}%)")
# --- The code continues below where it previously cut off ---
print(f"    • Non-virulent: {len(non_virulent_with_seqs):,} ({len(non_virulent_with_seqs)/len(novel_sequences)*100:.1f}%)")

if missing_sequences:
    print(f"    • Missing/Unmatched: {len(missing_sequences):,} ({len(missing_sequences)/len(novel_sequences)*100:.1f}%)")

print(f"\n📂 Output Files Generated:")
print(f"    • Report: {os.path.basename(report_file)}")
print(f"    • Statistics: {os.path.basename(stats_file)}")
print(f"    • Updated DB: {os.path.basename(DB_VIRULENT)}")
print(f"    • Updated Non-DB: {os.path.basename(DB_NON_VIRULENT)}")
print("\n" + "="*80)

In [ ]:
# ============================================================================
# CELL 5: Remove Internal Duplicates (Deduplication)
# ============================================================================
# @title Cell 5: Remove Internal Duplicates from Updated Databases

import os
from datetime import datetime
import shutil

print("="*80)
print("FINAL STEP: INTERNAL DUPLICATE REMOVAL")
print("="*80)

# ===== CONFIGURATION =====
# Ensure DB_BASE_PATH is defined (from previous cells)
if 'DB_BASE_PATH' not in locals():
    DB_BASE_PATH = "/content/drive/MyDrive/VirulentPred_Databases"

DB_VIRULENT = os.path.join(DB_BASE_PATH, "DB.fasta")
DB_NON_VIRULENT = os.path.join(DB_BASE_PATH, "Non-DB.fasta")

# ===== DEDUPLICATION FUNCTION =====
def remove_internal_duplicates(fasta_path):
    """
    Parses a FASTA file and removes sequences that have identical
    amino acid strings to a previously seen entry in the same file.
    """
    if not os.path.exists(fasta_path):
        print(f"❌ File not found: {fasta_path}")
        return

    filename = os.path.basename(fasta_path)
    print(f"\n🧹 Processing: {filename}")

    # 1. Create Backup
    backup_path = fasta_path + ".bak"
    shutil.copy2(fasta_path, backup_path)
    print(f"    Item backed up to: {os.path.basename(backup_path)}")

    unique_sequences = {} # key: sequence_string, value: header
    duplicate_count = 0
    total_read = 0

    # 2. Read and Filter
    # We read the backup to ensure we don't corrupt the file we are writing to
    with open(backup_path, 'r') as f_in:
        header = None
        seq_lines = []

        for line in f_in:
            line = line.strip()
            if line.startswith('>'):
                # Process previous sequence
                if header:
                    full_seq = "".join(seq_lines)
                    if full_seq in unique_sequences:
                        duplicate_count += 1
                        # Optional: Print duplicate info (commented out to reduce noise)
                        # print(f"      Duplicate found: {header} (Matches {unique_sequences[full_seq]})")
                    else:
                        unique_sequences[full_seq] = header
                    total_read += 1

                # Start new record
                header = line
                seq_lines = []
            else:
                seq_lines.append(line)

        # Process the very last sequence
        if header:
            full_seq = "".join(seq_lines)
            if full_seq in unique_sequences:
                duplicate_count += 1
            else:
                unique_sequences[full_seq] = header
            total_read += 1

    # 3. Write Cleaned Data back to original path
    with open(fasta_path, 'w') as f_out:
        for seq, head in unique_sequences.items():
            f_out.write(f"{head}\n{seq}\n")

    # 4. Report
    print(f"    📊 Statistics for {filename}:")
    print(f"       • Original Count: {total_read:,}")
    print(f"       • Duplicates Removed: {duplicate_count:,}")
    print(f"       • Final Unique Count: {len(unique_sequences):,}")

    return len(unique_sequences)

# ===== EXECUTION =====

print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 1. Clean DB.fasta (Virulent)
final_vir_count = remove_internal_duplicates(DB_VIRULENT)

# 2. Clean Non-DB.fasta (Non-Virulent)
final_non_vir_count = remove_internal_duplicates(DB_NON_VIRULENT)

# ===== FINAL PROJECT SUMMARY =====
print("\n" + "="*80)
print("PROJECT COMPLETED SUCCESSFULLY")
print("="*80)
print(f"The databases are now updated, merged, and cleaned of duplicates.")
print(f"They are ready for Model Retraining.")

print(f"\n📂 Final Database Locations:")
print(f"   1. {DB_VIRULENT} ({final_vir_count:,} seqs)")
print(f"   2. {DB_NON_VIRULENT} ({final_non_vir_count:,} seqs)")
print("="*80)

In [ ]:
# ============================================================================
# CELL 6: Generate Final Updated Statistics (Post-Deduplication)
# ============================================================================
# @title Cell 6: Update Statistics Report

import pandas as pd
import os
import glob
from datetime import datetime

print("="*80)
print("GENERATING FINAL STATISTICS REPORT")
print("="*80)

# ===== CONFIGURATION =====
if 'DB_BASE_PATH' not in locals():
    DB_BASE_PATH = "/content/drive/MyDrive/VirulentPred_Databases"

DB_VIRULENT = os.path.join(DB_BASE_PATH, "DB.fasta")
DB_NON_VIRULENT = os.path.join(DB_BASE_PATH, "Non-DB.fasta")

# ===== HELPER: COUNT SEQUENCES =====
def count_seqs_robust(fasta_file):
    if not os.path.exists(fasta_file): return 0
    with open(fasta_file, 'r') as f:
        return sum(1 for line in f if line.startswith('>'))

# ===== 1. FIND PREVIOUS STATS FILE =====
# We need this to get the "Novel Sequences" and "Prediction" counts
# without re-running the whole analysis.
stats_pattern = os.path.join(DB_BASE_PATH, "VirulentPred_Statistics_*.xlsx")
list_of_files = glob.glob(stats_pattern)

if not list_of_files:
    print("⚠️ No previous statistics file found. Creating a fresh count...")
    # Fallback if no file exists
    prev_stats = {}
else:
    # Get the latest file
    latest_file = max(list_of_files, key=os.path.getctime)
    print(f"\n📂 Reading previous stats from: {os.path.basename(latest_file)}")

    # Read the 'Statistics' sheet
    df_prev = pd.read_excel(latest_file, sheet_name='Statistics')
    # Convert to dictionary for easy access
    prev_stats = dict(zip(df_prev['Metric'], df_prev['Value']))

# ===== 2. GET CURRENT (FINAL) COUNTS =====
final_vir_count = count_seqs_robust(DB_VIRULENT)
final_nonvir_count = count_seqs_robust(DB_NON_VIRULENT)

# ===== 3. CALCULATE METRICS =====
# Try to get 'After Update' counts (which are now 'Pre-Deduplication') from the file
# If not found, we assume 0 or handle gracefully
pre_dedup_vir = prev_stats.get('DB.fasta (After)', final_vir_count)
pre_dedup_nonvir = prev_stats.get('Non-DB.fasta (After)', final_nonvir_count)

# Ensure they are integers (handles cases where Excel might have stored them as strings)
try:
    pre_dedup_vir = int(pre_dedup_vir)
    pre_dedup_nonvir = int(pre_dedup_nonvir)
except:
    pass

duplicates_vir = pre_dedup_vir - final_vir_count
duplicates_nonvir = pre_dedup_nonvir - final_nonvir_count

# ===== 4. BUILD FINAL DATAFRAME =====
final_data = {
    'Metric': [
        '--- INPUT DATA ---',
        'Total Novel Sequences',
        'Virulent Predictions',
        'Non-virulent Predictions',
        'Missing/Unmatched',
        '--- DATABASE MERGE (Step 5) ---',
        'DB.fasta (Before Merge)',
        'DB.fasta (After Merge)',
        'Non-DB.fasta (Before Merge)',
        'Non-DB.fasta (After Merge)',
        '--- DEDUPLICATION (Step 6) ---',
        'Duplicates Removed (DB.fasta)',
        'Duplicates Removed (Non-DB.fasta)',
        '--- FINAL STATUS ---',
        'DB.fasta (Final Count)',
        'Non-DB.fasta (Final Count)',
        'Total Final Database Size'
    ],
    'Value': [
        '',
        prev_stats.get('Total Novel Sequences', 'N/A'),
        prev_stats.get('Virulent Predictions', 'N/A'),
        prev_stats.get('Non-virulent Predictions', 'N/A'),
        prev_stats.get('Missing Predictions', 'N/A'),
        '',
        prev_stats.get('DB.fasta (Before)', 'N/A'),
        pre_dedup_vir,
        prev_stats.get('Non-DB.fasta (Before)', 'N/A'),
        pre_dedup_nonvir,
        '',
        duplicates_vir,
        duplicates_nonvir,
        '',
        final_vir_count,
        final_nonvir_count,
        final_vir_count + final_nonvir_count
    ]
}

df_final = pd.DataFrame(final_data)

# ===== 5. SAVE FINAL REPORT =====
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
final_report_name = f"VirulentPred_Final_Statistics_{timestamp}.xlsx"
final_report_path = os.path.join(DB_BASE_PATH, final_report_name)

# We also want to preserve the detailed "All_Results" sheet from the previous file
with pd.ExcelWriter(final_report_path) as writer:
    df_final.to_excel(writer, sheet_name='Final_Statistics', index=False)

    # Copy the detailed report if it exists
    if list_of_files:
        try:
            df_details = pd.read_excel(latest_file, sheet_name='All_Results')
            df_details.to_excel(writer, sheet_name='All_Results', index=False)
            print("✅ 'All_Results' sheet copied to new report.")
        except:
            print("⚠️ Could not copy 'All_Results' sheet (maybe it didn't exist).")

print(f"\n💾 FINAL REPORT SAVED: {final_report_name}")
print(f"📍 Location: {final_report_path}")
print("\n" + "="*80)
print(f"SUMMARY:")
print(f"Removed {duplicates_vir} duplicates from Virulent DB.")
print(f"Removed {duplicates_nonvir} duplicates from Non-Virulent DB.")
print("="*80)

In [ ]:
from google.colab import drive
import os

print("="*80)
print("GOOGLE DRIVE SETUP")
print("="*80)

drive.mount('/content/drive')
print("✅ Google Drive mounted")